[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/05_cnn/05_cnn_solutions.ipynb)

# 05. CNN — 연습 문제 해설

[05_cnn.ipynb](05_cnn.ipynb) 끝의 연습 문제 4개에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

> **읽는 법** — 먼저 직접 풀어본 뒤 보세요. 셀은 위에서부터 순서대로 실행해야 하고,
> 실행 결과는 저장되어 있지 않으니 직접 실행해야 출력이 나타납니다.

> **아래 준비 셀은 네 부분입니다.**
> 1. Colab이면 패키지 설치
> 2. 라이브러리 import와 연산 장치(`device`) 결정
> 3. MNIST 데이터 로드와 `DataLoader` 구성
> 4. 본문과 동일한 `evaluate()` / `train_model()` 함수 정의
>
> 본문 05번을 이미 실행했다면 내용이 같습니다. **그대로 실행**하고 아래 문제로 넘어가세요.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q torch torchvision matplotlib

# ══════════════════════════════════════════════════════════════
# ① 라이브러리와 연산 장치 — GPU가 있으면 cuda, 없으면 cpu
# ══════════════════════════════════════════════════════════════
import time
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print("device:", device)

# ══════════════════════════════════════════════════════════════
# ② 데이터 준비 — MNIST 다운로드와 DataLoader (본문 05번과 동일)
# ══════════════════════════════════════════════════════════════
# ToTensor: 이미지를 0~1 사이 실수 텐서로 바꾼다
transform = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.MNIST(root="../../../data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="../../../data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

# ══════════════════════════════════════════════════════════════
# ③ 평가 함수 — 정확도 계산
# ══════════════════════════════════════════════════════════════
def evaluate(model, loader):
    model.eval()          # 평가 모드 (Dropout·BatchNorm 동작이 바뀐다)
    correct, total = 0, 0
    with torch.no_grad():  # 기울기 계산 끄기
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.size(0)
    return correct / total

# ══════════════════════════════════════════════════════════════
# ④ 학습 함수 — 정확도뿐 아니라 소요 시간과 파라미터 수까지 돌려준다
#    (MLP와 CNN을 '성능·속도·크기' 세 축으로 비교하기 위함)
# ══════════════════════════════════════════════════════════════
def train_model(model, epochs=3, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            # 학습 4단계: 기울기 초기화 -> 손실 -> 역전파 -> 갱신
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
    elapsed = time.time() - t0
    test_acc = evaluate(model, test_loader)
    n_params = sum(p.numel() for p in model.parameters())   # 학습되는 값의 총 개수
    return test_acc, elapsed, n_params

## 연습 1. 같은 3 epoch 기준, 04번 MLP vs CNN 정확도 비교

In [ ]:
class MnistMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # MaxPool2d(2): 2×2 구역에서 최댓값만 남겨 크기를 절반으로 줄인다
            # Conv2d(입력채널, 출력채널, 커널크기): padding=1이면 출력 크기가 입력과 같게 유지된다
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(0)
mlp = MnistMLP().to(device)
acc_mlp, time_mlp, params_mlp = train_model(mlp, epochs=3)

torch.manual_seed(0)
cnn = SimpleCNN().to(device)
acc_cnn, time_cnn, params_cnn = train_model(cnn, epochs=3)

print(f"{'모델':<8}{'test_acc':>10}{'학습시간(s)':>14}{'파라미터수':>14}")
print(f"{'MLP':<8}{acc_mlp:>10.4f}{time_mlp:>14.1f}{params_mlp:>14,}")
print(f"{'CNN':<8}{acc_cnn:>10.4f}{time_cnn:>14.1f}{params_cnn:>14,}")

**해설**
- 같은 3 epoch, 비슷한 파라미터 규모에서도 CNN이 MLP보다 test accuracy가 더 높게 나오는 경우가 많습니다.
- 이유는 CNN이 "공간적으로 가까운 픽셀은 서로 관련이 있다"는 이미지의 구조적 특성(inductive bias)을 필터/파라미터 공유를 통해 이미 반영하고 있기 때문입니다. MLP는 이미지를 1차원으로 펼치면서 이 구조 정보를 잃어버립니다.
- 반면 CNN은 Convolution 연산 때문에 같은 epoch 기준 학습 시간이 MLP보다 더 걸릴 수 있습니다 — 정확도와 속도는 트레이드오프입니다.

## 연습 2. Conv 채널 수 / 층 수를 늘리면?

In [ ]:
class DeeperCNN(nn.Module):
    """채널 16->32 를 32->64로 늘리고, conv 블록을 하나 더 추가한 버전"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28->14
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14->7
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(),                    # 7->7 (추가 블록)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(0)
deeper_cnn = DeeperCNN().to(device)
acc_deep, time_deep, params_deep = train_model(deeper_cnn, epochs=3)

print(f"{'모델':<12}{'test_acc':>10}{'학습시간(s)':>14}{'파라미터수':>14}")
print(f"{'CNN(기본)':<12}{acc_cnn:>10.4f}{time_cnn:>14.1f}{params_cnn:>14,}")
print(f"{'CNN(확장)':<12}{acc_deep:>10.4f}{time_deep:>14.1f}{params_deep:>14,}")

**해설**
- 채널 수(16→32, 32→64)와 층 수를 늘리면 모델이 더 다양하고 복잡한 특징을 표현할 수 있어 test accuracy가 조금 더 오르는 경향이 있습니다.
- 다만 파라미터 수와 학습 시간이 함께 늘어나므로, 정확도 향상 폭이 크지 않다면 "더 큰 모델"이 항상 정답은 아닙니다.
- MNIST처럼 비교적 쉬운 데이터셋에서는 이미 기본 CNN도 99% 근처에 도달하기 때문에 개선 폭이 작게 보일 수 있습니다 — 더 어려운 데이터셋(CIFAR-10 등)에서 이 트레이드오프가 훨씬 뚜렷하게 드러납니다.

## 연습 3. 필터를 가로 경계선 탐지기로 바꾸면?

본문 2절의 세로 경계선 필터는 행과 열을 뒤집으면(전치) 그대로 가로 경계선 탐지기가 됩니다.
**위아래로 나뉜 이미지**에 두 필터를 각각 적용해서 무엇이 달라지는지 봅니다.

In [ ]:
# 위 절반이 검고 아래 절반이 흰 8×8 이미지
img_horiz_split = torch.zeros(1, 1, 8, 8)
img_horiz_split[:, :, 4:, :] = 1.0

vertical_edge_filter = torch.tensor([[[[-1., 0., 1.],
                                       [-1., 0., 1.],
                                       [-1., 0., 1.]]]])
# transpose(2, 3): 마지막 두 축(행·열)을 뒤집는다 -> 가로 경계선 탐지기
horizontal_edge_filter = vertical_edge_filter.transpose(2, 3)

print("가로 경계선 필터:")
print(horizontal_edge_filter[0, 0].numpy())


def apply_filter(image, filt):
    conv = nn.Conv2d(1, 1, kernel_size=3, bias=False)
    with torch.no_grad():
        conv.weight.copy_(filt)
    return conv(image)[0, 0].detach().numpy().round(1)


print("\n[세로 경계선 필터]를 위아래로 나뉜 이미지에 적용:")
print(apply_filter(img_horiz_split, vertical_edge_filter))

print("\n[가로 경계선 필터]를 위아래로 나뉜 이미지에 적용:")
print(apply_filter(img_horiz_split, horizontal_edge_filter))

**해설**

- **세로 필터는 전부 0**입니다. 이 이미지에는 세로 경계선이 하나도 없기 때문입니다.
  필터가 "왼쪽과 오른쪽의 밝기 차이"를 재는데, 이 이미지는 어느 행에서든 왼쪽과 오른쪽이 같습니다.
- **가로 필터는 경계가 있던 두 행에서만 3**이 나옵니다. 본문 2절에서 세로 경계선 이미지에
  세로 필터를 썼을 때와 정확히 같은 모양이, 90도 돌아간 채로 나타납니다.

**여기서 얻을 것**: 필터 하나는 **한 가지 특징만** 봅니다. 세로 경계선 탐지기는 가로 경계선을
전혀 보지 못합니다. 그래서 실제 CNN은 본문 4절처럼 `Conv2d(1, 16, ...)`으로 **필터를 16개씩
동시에** 둡니다. 각자 다른 특징을 맡게 하려는 것이고, 무엇을 맡을지는 사람이 정해주지 않고
학습이 알아서 나눕니다.

## 연습 4. `MaxPool2d(2)`를 `MaxPool2d(3)`으로 바꾸면?

본문 3절의 이동 실험을 풀링 구역 크기만 바꿔서 다시 돌려봅니다.

In [ ]:
conv = nn.Conv2d(1, 1, kernel_size=3, bias=False)
with torch.no_grad():
    conv.weight.copy_(vertical_edge_filter)


def bar_image(col):
    im = torch.zeros(1, 1, 8, 8)
    im[:, :, :, col] = 1.0
    return im


for name, pool in [("MaxPool2d(2)", nn.MaxPool2d(2)), ("MaxPool2d(3)", nn.MaxPool2d(3))]:
    print(f"\n[{name}]")
    for shift in range(4):
        fm = conv(bar_image(3 + shift))
        pl = pool(fm)
        print(f"  {shift}칸 이동  특성 맵: {fm[0, 0, 0].detach().numpy().round(0)}"
              f"   풀링 후: {pl[0, 0, 0].detach().numpy().round(0)}")

**해설**

출력을 정리하면 이렇습니다.

| 이동 | `MaxPool2d(2)` 결과 | `MaxPool2d(3)` 결과 |
|---|---|---|
| 0칸 | `[3, 0, 0]` | `[3, 0]` |
| 1칸 | `[0, 3, 0]` | `[3, 0]` |
| 2칸 | `[0, 3, 0]` | `[0, 3]` |
| 3칸 | `[0, 0, 3]` | `[0, 3]` |

- **구역이 2일 때**는 1칸과 2칸만 같았습니다. **구역이 3이 되면 0~1칸이 같아지고, 2~3칸이
  같아집니다.** 구역이 커진 만큼 더 큰 이동까지 흡수합니다.
- **더 얻는 것**: 위치 변화에 더 둔감해지고, 출력 크기가 더 줄어 계산이 더 싸집니다
  (6×6 → 2×2로, 9분의 1).
- **더 잃는 것**: 위치 정보가 더 거칠어집니다. 3칸 안의 차이는 아예 구별할 수 없게 됩니다.
  숫자 인식에서 획의 위치가 3픽셀씩 뭉개지면 `6`과 `8`처럼 위치로 구별하는 것들이 헷갈리기
  시작합니다. 게다가 크기가 너무 빨리 줄어서 층을 깊게 쌓을 여지가 사라집니다.

**그래서 실무에서는 대부분 2를 씁니다.** 한 번에 크게 줄이는 것보다, 2씩 여러 번 줄이면서
그 사이사이에 Conv를 끼워 넣는 쪽이 낫기 때문입니다.